**ETAPE 2 : scoring**


Ces deux premier code servent à avoir un apperçu des clients avec échecs de payements, selon la raison, afin de pouvoir effectuer un ratio
ces codes m'ont permis de corriger une erreur dans le premier notebook où "suceded" et "success" étaient considérés comme deux status différents ..

In [12]:
import pandas as pd

df_payments = pd.read_csv("payments_clean_corrigé.csv")
print(df_payments['status'].value_counts())

status
succeeded    3980
failed       1984
pending       581
refunded      375
disputed      216
cancelled     141
Name: count, dtype: int64


In [13]:
print("💳 ANALYSE DES PAIEMENTS PAR SUBSCRIBER\n")

# Toutes les valeurs de status présentes
print("Statuts présents dans payments :")
print(df_payments['status'].value_counts())

print("\n--- DÉTAIL PAR SUBSCRIBER ---")

paiements_detail = df_payments.groupby('user_id').agg(
    total          = ('status', 'count'),
    succeeded      = ('status', lambda x: (x == 'succeeded').sum()),
    failed         = ('status', lambda x: (x == 'failed').sum()),
    pending        = ('status', lambda x: (x == 'pending').sum()),
    refunded       = ('status', lambda x: (x == 'refunded').sum()),
    disputed       = ('status', lambda x: (x == 'disputed').sum()),
    cancelled      = ('status', lambda x: (x == 'cancelled').sum()),
).reset_index()

# Calcul du taux d'échec
paiements_detail['taux_echec_pct'] = (
    paiements_detail['failed'] / paiements_detail['total'] * 100
).round(1)

print(f"\nTop 10 subscribers avec le plus d'échecs :")
print(
    paiements_detail
    .sort_values('failed', ascending=False)
    .head(10)
    .to_string()
)

💳 ANALYSE DES PAIEMENTS PAR SUBSCRIBER

Statuts présents dans payments :
status
succeeded    3980
failed       1984
pending       581
refunded      375
disputed      216
cancelled     141
Name: count, dtype: int64

--- DÉTAIL PAR SUBSCRIBER ---

Top 10 subscribers avec le plus d'échecs :
     user_id  total  succeeded  failed  pending  refunded  disputed  cancelled  taux_echec_pct
323      793     41         23      11        4         0         2          1            26.8
704     1728     37         23      11        2         0         1          0            29.7
611     1487     24         11      11        0         0         0          2            45.8
22        57     22          7      11        0         4         0          0            50.0
290      704     29         15      10        3         0         1          0            34.5
696     1705     25         15      10        0         0         0          0            40.0
219      538     17          6       9        

On attribue un score de risque à chaque suscribers, ce score est compris entre 0 et 100 où 0 correspond à un suscriber sans risque et 100 correspond à un suscriber à risque.
Ce score est calculé en sommant les points qu'on associe à chaque feature qu'un suscriber peut avoir en se basant sur leur importance "à risque".

- `payments.status` (= table payments, colonne status)
représente un risque très élevé. On calcule le ratio :
`nb paiements failed / nb paiements total`: plus ce taux est élevé, plus le subscriber est risqué.
Un taux de 100% d'échecs donne **30 points**, un taux de 50% donne 15 points.

- `payments.stripe_error_code`
Si Stripe a détecté une carte volée (`stolen_card`)on a un risque élevé de fraude. Ce signal est binaire (présent ou absent).
On attribue **25 points** dès qu'au moins un paiement contient ce code d'erreur.

- `memberships.reason`
Si un subscriber a été retiré d'un abonnement pour raison `fraud` ou `payment_failed`, c'est un signal confirmé par la plateforme.
->`fraud`= **20 points**    
-> `payment_failed`= **10 points**   
-> Autre raison = **0 point**

- `complaints.target_id`
On compte le nombre de fois où ce subscriber apparaît dans les plaintes d'une réclamation.
-> 0 plainte = **0 points**
-> 1 plainte = **4 points**
-> 2 plaintes = **7 points**
->3 plaintes ou plus = **10 points**

- `users.prefix_flag`
Un préfixe téléphonique qui ne correspond pas au pays déclaré peut indiquer une fausse identité/ un risque de fraude.
-> `mismatch` = **8 points**
-> `unknown` ou `country_not_mapped` = **3 points**
-> `ok` = **0 points**

- `complaints.reporter_id` et `complaints.target_id`
Un utilisateur qui se plaint de lui-même (`reporter_id == target_id`)
peut representer une tentative de manipulation du système de réclamations.
-> Au moins 1 self-complaint = **4 points**
-> Aucun = **0 points**

- `users.last_seen_clean`
Un subscriber inactif depuis plus de 6 mois avec des signaux de risque passés reste potentiellement dangereux.
-> Inactif depuis plus de 6 mois = **3 points**
-> Actif récemment = **0 points**

- Nouveau subscriber (0 paiement) | Score de base 30/100 — absence d'historique = risque modéré par défaut
- 1 seul paiement réussi | Taux d'échec = 0%, mais score de base 15/100 — historique insuffisant
- Subscriber inactif depuis 6 mois | +**3 points** sur le score final
- Subscriber sans aucune réclamation | **0 point** sur cette feature
- Subscriber banni (status = banned) | Score forcé à 100 — décision déjà prise par la plateforme

In [14]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

df_users = pd.read_csv("users_clean_corrigé.csv")
df_payments = pd.read_csv("payments_clean_corrigé.csv")
df_memberships = pd.read_csv("memberships_clean_corrigé.csv")
df_complaints = pd.read_csv("complaints_clean_corrigé.csv")

df_users['last_seen_clean'] = pd.to_datetime(df_users['last_seen_clean'], errors='coerce')
df_payments['created_at_clean'] = pd.to_datetime(df_payments['created_at_clean'], errors='coerce')
df_memberships['joined_at_clean'] = pd.to_datetime(df_memberships['joined_at_clean'], errors='coerce')

print(f" users : {len(df_users)} lignes")
print(f" payments : {len(df_payments)} lignes")
print(f" memberships : {len(df_memberships)} lignes")
print(f" complaints : {len(df_complaints)} lignes")

 users : 2001 lignes
 payments : 7277 lignes
 memberships : 1083 lignes
 complaints : 1211 lignes


In [15]:
# Date de référence pour le calcul d'inactivité
TODAY = datetime(2025, 6, 1)
SEUIL_INACTIVITE = TODAY - timedelta(days=180)

# Liste de tous les subscribers (users qui ont au moins un membership)
subscribers = df_memberships['user_id'].unique()
print(f"Nombre de subscribers à scorer : {len(subscribers)}")

# ── FEATURE 1 : Taux d'échecs de paiement ─────────────────
pay_stats = df_payments.groupby('user_id').agg(
    total_payments = ('status', 'count'),
    failed_payments = ('status', lambda x: (x == 'failed').sum()),
    has_stolen_card = ('stripe_error_code', lambda x: ('stolen_card' in x.values)),
).reset_index()
pay_stats['taux_echec'] = pay_stats['failed_payments'] / pay_stats['total_payments']

# ── FEATURE 2 : Raison de sortie membership ───────────────
mem_stats = df_memberships.groupby('user_id').agg(
    has_fraud_reason = ('reason', lambda x: 'fraud' in x.values),
    has_payment_reason = ('reason', lambda x: 'payment_failed' in x.values),
    nb_memberships = ('id', 'count'),
).reset_index()

# ── FEATURE 3 : Plaintes reçues ───────────────────────────
complaints_received = df_complaints.groupby('target_id').agg(
    nb_plaintes = ('id', 'count')
).reset_index().rename(columns={'target_id': 'user_id'})

# ── FEATURE 4 : Self-complaints ───────────────────────────
self_complaints = df_complaints[
    df_complaints['reporter_id'] == df_complaints['target_id']
].groupby('target_id').agg(
    nb_self_complaints = ('id', 'count')
).reset_index().rename(columns={'target_id': 'user_id'})

# ── FEATURE 5 : Inactivité + prefix_flag ──────────────────

# Recréer status_clean depuis status (mapping des codes numériques)
mapping_user_status = {
     0.0: "active",
     1.0: "inactive",
     2.0: "suspended",
     3.0: "pending",
     4.0: "verified",
    -1.0: "banned",
    99.0: "deleted",
}
df_users['status_clean'] = df_users['status'].map(mapping_user_status).fillna("unknown")

user_stats = df_users[['id', 'last_seen_clean', 'prefix_flag', 'status_clean']].copy()
user_stats = user_stats.rename(columns={'id': 'user_id'})
user_stats['is_inactive'] = user_stats['last_seen_clean'] < SEUIL_INACTIVITE
user_stats['is_banned']   = user_stats['status_clean'] == 'banned'

print("✅ Features calculées")

Nombre de subscribers à scorer : 830
✅ Features calculées


In [16]:
# Partir de la liste des subscribers
scores = pd.DataFrame({'user_id': subscribers})

# Joindre toutes les features
scores = scores.merge(pay_stats, on='user_id', how='left')
scores = scores.merge(mem_stats, on='user_id', how='left')
scores = scores.merge(complaints_received, on='user_id', how='left')
scores = scores.merge(self_complaints, on='user_id', how='left')
scores = scores.merge(user_stats, on='user_id', how='left')

# Remplir les valeurs manquantes (subscriber sans historique = 0)
scores['total_payments'] = scores['total_payments'].fillna(0)
scores['failed_payments'] = scores['failed_payments'].fillna(0)
scores['taux_echec'] = scores['taux_echec'].fillna(0)
scores['has_stolen_card'] = scores['has_stolen_card'].fillna(False)
scores['has_fraud_reason'] = scores['has_fraud_reason'].fillna(False)
scores['has_payment_reason'] = scores['has_payment_reason'].fillna(False)
scores['nb_plaintes'] = scores['nb_plaintes'].fillna(0)
scores['nb_self_complaints'] = scores['nb_self_complaints'].fillna(0)
scores['is_inactive'] = scores['is_inactive'].fillna(False)
scores['is_banned'] = scores['is_banned'].fillna(False)
scores['prefix_flag'] = scores['prefix_flag'].fillna('unknown')

# ── CALCUL DU SCORE ───────────────────────────────────────

# Feature 1 — Taux échec paiement (30 pts max)
scores['score_echec'] = scores.apply(lambda r:
    0 if r['total_payments'] == 0 # nouveau subscriber
    else round(r['taux_echec'] * 30),
axis=1)

# Feature 2 — Carte volée (25 pts)
scores['score_stolen'] = scores['has_stolen_card'].apply(
    lambda x: 25 if x else 0
)

# Feature 3 — Raison membership fraud (20 pts max)
scores['score_fraud_reason'] = scores.apply(lambda r:
    20 if r['has_fraud_reason']
    else (10 if r['has_payment_reason'] else 0),
axis=1)

# Feature 4 — Plaintes reçues (10 pts max)
scores['score_plaintes'] = scores['nb_plaintes'].apply(
    lambda n: 0 if n == 0 else (4 if n == 1 else (7 if n == 2 else 10))
)

# Feature 5 — Mismatch pays/téléphone (8 pts max)
scores['score_prefix'] = scores['prefix_flag'].apply(
    lambda f: 8 if f == 'mismatch' else (3 if f in ['unknown', 'country_not_mapped'] else 0)
)

# Feature 6 — Self-complaints (4 pts)
scores['score_self'] = scores['nb_self_complaints'].apply(
    lambda n: 4 if n > 0 else 0
)

# Feature 7 — Inactivité (3 pts)
scores['score_inactivite'] = scores['is_inactive'].apply(
    lambda x: 3 if x else 0
)

# ── SCORE FINAL ───────────────────────────────────────────
scores['risk_score'] = (
    scores['score_echec'] +
    scores['score_stolen'] +
    scores['score_fraud_reason'] +
    scores['score_plaintes'] +
    scores['score_prefix'] +
    scores['score_self'] +
    scores['score_inactivite']
).clip(0, 100)  # on s'assure que le score reste entre 0 et 100

# Edge case — subscribers bannis → score forcé à 100
scores.loc[scores['is_banned'] == True,'risk_score'] = 100

# Edge case — nouveau subscriber (0 paiement) → score de base 30
scores.loc[scores['total_payments'] == 0,'risk_score'] = scores.loc[
    scores['total_payments'] == 0, 'risk_score'
].apply(lambda x: max(x, 30))

print("✅ Scores calculés")
print(f"\nDistribution des scores :")
print(scores['risk_score'].describe().round(1))

✅ Scores calculés

Distribution des scores :
count    830.0
mean      22.8
std       16.8
min        0.0
25%       11.0
50%       18.0
75%       33.0
max      100.0
Name: risk_score, dtype: float64


/tmp/ipykernel_5351/1792045743.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scores['has_stolen_card'] = scores['has_stolen_card'].fillna(False)


In [17]:
# Sélection des colonnes finales
output = scores[[
    'user_id', 'risk_score',
    'total_payments', 'taux_echec', 'has_stolen_card',
    'has_fraud_reason', 'nb_plaintes', 'nb_self_complaints',
    'prefix_flag', 'is_inactive', 'is_banned'
]].sort_values('risk_score', ascending=False)

# Export CSV
output.to_csv('scored.csv', index=False)

print("✅ scored.csv exporté !")
print(f"\nTop 10 subscribers les plus à risque :")
print(output.head(10).to_string())

# Téléchargement
from google.colab import files
files.download('scored.csv')

✅ scored.csv exporté !

Top 10 subscribers les plus à risque :
     user_id  risk_score  total_payments  taux_echec  has_stolen_card  has_fraud_reason  nb_plaintes  nb_self_complaints prefix_flag  is_inactive  is_banned
1        690         100             9.0    0.444444            False              True          0.0                 0.0          ok         True       True
199      852         100            18.0    0.388889             True             False          1.0                 0.0     unknown        False       True
189     1502         100             3.0    0.333333            False             False          1.0                 0.0          ok        False       True
270      717         100            13.0    0.307692            False             False          1.0                 0.0          ok         True       True
417      461         100             5.0    0.400000            False             False          1.0                 0.0          ok         True       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>